# Logical Operations (AND, OR, XOR)

This notebook demonstrates training neural networks to learn logical gates.

- **AND / OR**: Linearly separable - can be learned with a single layer
- **XOR**: NOT linearly separable - requires hidden layers

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '../src')

from neura_command import Sequential, Dense, ReLU, Sigmoid, Adam, MSELoss

np.random.seed(42)

## 1. Define the Logic Gates

In [ ]:
# Input combinations
X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1],
], dtype=np.float32)

# Outputs for each gate
y_and = np.array([[0], [0], [0], [1]], dtype=np.float32)
y_or  = np.array([[0], [1], [1], [1]], dtype=np.float32)
y_xor = np.array([[0], [1], [1], [0]], dtype=np.float32)

# Replicate data for training (small dataset)
n_repeats = 100
X_train = np.tile(X, (n_repeats, 1))
y_and_train = np.tile(y_and, (n_repeats, 1))
y_or_train = np.tile(y_or, (n_repeats, 1))
y_xor_train = np.tile(y_xor, (n_repeats, 1))

print(f"Training samples per gate: {len(X_train)}")

## 2. AND Gate

AND is linearly separable - a single layer (perceptron) can learn it.

In [ ]:
# Create simple model for AND
model_and = Sequential(
    layers=[Dense(2, 1), Sigmoid()],
    optimizer=Adam(learning_rate=0.5),
    loss=MSELoss(),
)

# Train
model_and.train(X_train, y_and_train, epochs=50, batch_size=16, verbose=False)

# Test
predictions = model_and.predict(X)
binary_preds = (predictions > 0.5).astype(int)

print("AND Gate Results:")
print("-" * 40)
for i in range(4):
    x1, x2 = int(X[i, 0]), int(X[i, 1])
    pred = binary_preds[i, 0]
    true = int(y_and[i, 0])
    status = "✓" if pred == true else "✗"
    print(f"{x1} AND {x2} = {pred} (expected: {true}) {status}")

accuracy = np.mean(binary_preds == y_and) * 100
print(f"\nAccuracy: {accuracy:.0f}%")

## 3. OR Gate

In [ ]:
# Create simple model for OR
model_or = Sequential(
    layers=[Dense(2, 1), Sigmoid()],
    optimizer=Adam(learning_rate=0.5),
    loss=MSELoss(),
)

# Train
model_or.train(X_train, y_or_train, epochs=50, batch_size=16, verbose=False)

# Test
predictions = model_or.predict(X)
binary_preds = (predictions > 0.5).astype(int)

print("OR Gate Results:")
print("-" * 40)
for i in range(4):
    x1, x2 = int(X[i, 0]), int(X[i, 1])
    pred = binary_preds[i, 0]
    true = int(y_or[i, 0])
    status = "✓" if pred == true else "✗"
    print(f"{x1} OR {x2} = {pred} (expected: {true}) {status}")

accuracy = np.mean(binary_preds == y_or) * 100
print(f"\nAccuracy: {accuracy:.0f}%")

## 4. XOR Gate

XOR is **not** linearly separable! A single perceptron cannot learn it.
We need at least one hidden layer.

In [ ]:
# First, let's try with a single layer (this will FAIL)
model_xor_simple = Sequential(
    layers=[Dense(2, 1), Sigmoid()],
    optimizer=Adam(learning_rate=0.5),
    loss=MSELoss(),
)

model_xor_simple.train(X_train, y_xor_train, epochs=100, batch_size=16, verbose=False)

predictions = model_xor_simple.predict(X)
binary_preds = (predictions > 0.5).astype(int)

print("XOR Gate (Single Layer - Expected to FAIL):")
print("-" * 40)
for i in range(4):
    x1, x2 = int(X[i, 0]), int(X[i, 1])
    pred = binary_preds[i, 0]
    true = int(y_xor[i, 0])
    status = "✓" if pred == true else "✗"
    print(f"{x1} XOR {x2} = {pred} (expected: {true}) {status}")

accuracy = np.mean(binary_preds == y_xor) * 100
print(f"\nAccuracy: {accuracy:.0f}% (cannot learn XOR!)")

In [ ]:
# Now with a hidden layer (this WILL work)
model_xor = Sequential(
    layers=[
        Dense(2, 8),    # 2 inputs -> 8 hidden units
        ReLU(),
        Dense(8, 1),    # 8 hidden -> 1 output
        Sigmoid(),
    ],
    optimizer=Adam(learning_rate=0.1),
    loss=MSELoss(),
)

model_xor.train(X_train, y_xor_train, epochs=200, batch_size=16, verbose=False)

predictions = model_xor.predict(X)
binary_preds = (predictions > 0.5).astype(int)

print("XOR Gate (With Hidden Layer):")
print("-" * 40)
for i in range(4):
    x1, x2 = int(X[i, 0]), int(X[i, 1])
    pred = binary_preds[i, 0]
    true = int(y_xor[i, 0])
    status = "✓" if pred == true else "✗"
    print(f"{x1} XOR {x2} = {pred} (expected: {true}) {status}")

accuracy = np.mean(binary_preds == y_xor) * 100
print(f"\nAccuracy: {accuracy:.0f}%")

## 5. Visualize Decision Boundaries

In [ ]:
def plot_decision_boundary(model, X, y, title):
    """Plot decision boundary for a 2D input space."""
    # Create mesh grid
    xx, yy = np.meshgrid(
        np.linspace(-0.5, 1.5, 100),
        np.linspace(-0.5, 1.5, 100)
    )
    
    # Get predictions for mesh
    mesh_input = np.c_[xx.ravel(), yy.ravel()].astype(np.float32)
    Z = model.predict(mesh_input)
    Z = Z.reshape(xx.shape)
    
    # Plot
    plt.contourf(xx, yy, Z, levels=50, cmap='RdBu', alpha=0.8)
    plt.colorbar(label='Output')
    
    # Plot decision boundary
    plt.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)
    
    # Plot data points
    colors = ['red' if yi == 0 else 'blue' for yi in y.flatten()]
    plt.scatter(X[:, 0], X[:, 1], c=colors, s=200, edgecolors='black', linewidths=2, zorder=5)
    
    # Add labels
    for i in range(len(X)):
        plt.annotate(f'{int(y[i, 0])}', (X[i, 0], X[i, 1]), 
                     ha='center', va='center', fontsize=12, fontweight='bold', color='white')
    
    plt.xlabel('Input 1')
    plt.ylabel('Input 2')
    plt.title(title)
    plt.xlim(-0.5, 1.5)
    plt.ylim(-0.5, 1.5)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

plt.subplot(1, 3, 1)
plot_decision_boundary(model_and, X, y_and, 'AND Gate')

plt.subplot(1, 3, 2)
plot_decision_boundary(model_or, X, y_or, 'OR Gate')

plt.subplot(1, 3, 3)
plot_decision_boundary(model_xor, X, y_xor, 'XOR Gate (with hidden layer)')

plt.tight_layout()
plt.show()

## 6. NAND and NOR Gates

In [ ]:
# NAND = NOT AND
y_nand = 1 - y_and
y_nand_train = np.tile(y_nand, (n_repeats, 1))

model_nand = Sequential(
    layers=[Dense(2, 1), Sigmoid()],
    optimizer=Adam(learning_rate=0.5),
    loss=MSELoss(),
)
model_nand.train(X_train, y_nand_train, epochs=50, batch_size=16, verbose=False)

predictions = (model_nand.predict(X) > 0.5).astype(int)

print("NAND Gate:")
for i in range(4):
    x1, x2 = int(X[i, 0]), int(X[i, 1])
    pred = predictions[i, 0]
    true = int(y_nand[i, 0])
    status = "✓" if pred == true else "✗"
    print(f"{x1} NAND {x2} = {pred} (expected: {true}) {status}")

In [ ]:
# NOR = NOT OR
y_nor = 1 - y_or
y_nor_train = np.tile(y_nor, (n_repeats, 1))

model_nor = Sequential(
    layers=[Dense(2, 1), Sigmoid()],
    optimizer=Adam(learning_rate=0.5),
    loss=MSELoss(),
)
model_nor.train(X_train, y_nor_train, epochs=50, batch_size=16, verbose=False)

predictions = (model_nor.predict(X) > 0.5).astype(int)

print("NOR Gate:")
for i in range(4):
    x1, x2 = int(X[i, 0]), int(X[i, 1])
    pred = predictions[i, 0]
    true = int(y_nor[i, 0])
    status = "✓" if pred == true else "✗"
    print(f"{x1} NOR {x2} = {pred} (expected: {true}) {status}")

## Summary

| Gate | Linearly Separable? | Layers Needed |
|------|--------------------|--------------|
| AND  | ✓ Yes | 1 (perceptron) |
| OR   | ✓ Yes | 1 (perceptron) |
| NAND | ✓ Yes | 1 (perceptron) |
| NOR  | ✓ Yes | 1 (perceptron) |
| XOR  | ✗ No  | 2+ (hidden layer required) |

This demonstrates the fundamental limitation of single-layer perceptrons (Minsky & Papert, 1969) and why we need **deep** neural networks!